# 🎬 مولّد فيديوهات الأطفال 3D (مجاني، على Colab)
يستعمل موديل مفتوح المصدر (Wan 2.1) ويولّد عدة مقاطع قصيرة ثم يدمجها في فيديو واحد عمودي (9:16).

**قبل البدء:** من القائمة ← *Runtime* ← *Change runtime type* ← اختر **T4 GPU**. ثم شغّل الخلايا بالترتيب.

⚠️ الجودة أقل من الأدوات التجارية، والمقاطع قصيرة (2–5 ثوانٍ). لكنه مجاني ويعمل على حسابك.

In [ ]:
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg ftfy

In [ ]:
# ====== عدّل هذه الخلية فقط ======
STYLE = "3D animated children's movie style, adorable big-eyed cartoon characters, vibrant colors, soft cinematic lighting, smooth motion, high detail"
SCENES = [
    "a cute baby with big blue eyes riding a friendly tiger on a sunny forest path with flowers and mountains",
    "a cute baby riding a smiling baby elephant through a colorful flower meadow, a wooden cottage in the background",
    "a cute baby riding a happy panda across a wooden bridge near a waterfall, pink blossom trees",
    "a cute baby riding a playful dolphin in sparkling blue water at sunset, splashing waves",
]
NUM_FRAMES = 49      # 49 ≈ 3 ثوانٍ، 81 ≈ 5 ثوانٍ (أبطأ ويحتاج ذاكرة أكثر)
FPS = 16
WIDTH, HEIGHT = 480, 832   # عمودي 9:16 (للأفقي بدّل القيمتين)
STEPS = 30
NEGATIVE = "blurry, low quality, deformed, extra limbs, text, watermark, ugly, static"

In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()   # يوفّر ذاكرة الكرت
print("جاهز")

In [ ]:
from diffusers.utils import export_to_video
import os
os.makedirs("clips", exist_ok=True)
for i, scene in enumerate(SCENES, 1):
    print(f"المقطع {i}/{len(SCENES)} ...")
    frames = pipe(prompt=f"{scene}. {STYLE}", negative_prompt=NEGATIVE,
                  height=HEIGHT, width=WIDTH, num_frames=NUM_FRAMES,
                  num_inference_steps=STEPS, guidance_scale=5.0).frames[0]
    export_to_video(frames, f"clips/clip_{i:02d}.mp4", fps=FPS)
    torch.cuda.empty_cache()
print("انتهت المقاطع")

In [ ]:
# دمج المقاطع في فيديو واحد وتحميله
with open("clips/list.txt", "w") as f:
    for name in sorted(os.listdir("clips")):
        if name.endswith(".mp4"): f.write(f"file '{name}'\n")
!ffmpeg -y -loglevel error -f concat -safe 0 -i clips/list.txt -c:v libx264 -pix_fmt yuv420p final.mp4
from google.colab import files
files.download("final.mp4")